In [ ]:

from datasets import load_dataset
import matplotlib as plt

In [ ]:

mnist_dataset = load_dataset("mnist")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

mnist/train-00000-of-00001.parquet:   0%|          | 0.00/15.6M [00:00<?, ?B/s]

mnist/test-00000-of-00001.parquet:   0%|          | 0.00/2.60M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/60000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

In [ ]:
mnist_dataset

DatasetDict({
    train: Dataset({
        features: ['image', 'label'],
        num_rows: 60000
    })
    test: Dataset({
        features: ['image', 'label'],
        num_rows: 10000
    })
})

In [ ]:
train_ds = mnist_dataset["train"]
example = train_ds[2]
print(example)
example["image"]

{'image': <PIL.PngImagePlugin.PngImageFile image mode=L size=28x28 at 0x7EBE4770D940>, 'label': 4}


In [ ]:
from torchvision import transforms
from torch.utils.data import DataLoader

tfm = transforms.ToTensor()

def preprocess(data):
  #data {image: .., label:..}
  data["pixel_values"] = [tfm(img) for img in data["image"]]
  return data

mnist = mnist_dataset.map(preprocess, batched=True, remove_columns=["image"])
mnist.set_format(type = "torch", columns=["pixel_values", "label"])

train_dataloader = DataLoader(mnist["train"], batch_size=256, shuffle=True)
test_dataloader = DataLoader(mnist["test"], batch_size=256, shuffle= True)

Map:   0%|          | 0/60000 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

In [ ]:
import torch.nn as nn
class LinearClassifier(nn.Module):
  def __init__(self):
    super(LinearClassifier, self).__init__()
    self.theta = nn.Linear(784, 10)

  def forward(self, x):
    x = x.flatten(1)
    result = self.theta(x)
    return result

In [ ]:
model = LinearClassifier()
print(model)


LinearClassifier(
  (theta): Linear(in_features=784, out_features=10, bias=True)
)


In [ ]:
import torch
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
criterion = nn.CrossEntropyLoss()

In [ ]:
epochs =2
for epoch in range(epochs):
  total_loss = 0
  for batch in train_dataloader:
    pixel_values = batch["pixel_values"]
    labels = batch['label']

    optimizer.zero_grad()
    outputs = model(pixel_values)
    loss = criterion(outputs, labels)
    total_loss += loss.item()

    loss.backward()
    optimizer.step()
  print(f"Epochs {epoch+1}, Loss: {total_loss:.4f}")
print("completed")

Epochs 1, Loss: 167.7990
Epochs 2, Loss: 100.7529
completed
